# Episode 29: Hyena: a GPT without attention, measured

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep029-hyena.ipynb)

We replace attention in our GPT with Hyena's long convolutions and gates, computed with the FFT, and measure it against our own attention models on speed, loss, long context and recall.

[The video](https://youtu.be/52TGLRyPcs8) · [The episode's page](https://mvgpt.nenuadrian.com/ep029-hyena/) · [The docs](https://nenuadrian.github.io/mvGPT/hyena/) · [The code at `ep029/end`](https://github.com/nenuadrian/mvGPT/tree/ep029/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep029/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt
!python setup.py -q build_ext --inplace  # the C++ BPE tokenizer, as a Python module

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))

## The tests
The FFT convolution against the sums written out, no output looking ahead, each channel as a
matrix, the filter's weights the same for any window, a hybrid's blocks, and writing through the
cache against running whole sequences.

In [ ]:
!python -m pytest -q tests/test_hyena.py

## A layer of each, timed
One attention layer against one Hyena layer, forward and backward, with as many tokens at each
length. The episode timed the Mac's GPU and a rented RTX 4090, where the two crossed at about
16,384 tokens; a T4 has no bfloat16 in hardware, so its times differ, and the longest lengths may
run out of memory, which the table says.

In [ ]:
!python hyena_bench.py --tokens 16384 --lengths 256 1024 4096 16384

## The data
FineWeb-Edu's pages, for training and for the long held-out pages the evaluation reads.

In [ ]:
!python data.py --web 1  # FineWeb-Edu: two files of about 2 GB, the web merges, the held-out pages and a file of tokens

## Small models
The episode trained six for half an hour each on a 4090 (`hyena_runs.sh`): 6 blocks of 384
channels with attention, with Hyena, and with Hyena but attention in block 3, on windows of
1,024 and 4,096. Here are attention and Hyena at 1,024, ten minutes each, to
`checkpoints/colab-attention` and `checkpoints/colab-hyena`.

In [ ]:
import torch

# bfloat16 in hardware: an L4 or A100 runtime has it, the free T4 (compute capability 7.5) doesn't.
AMPERE = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
print("bfloat16 in hardware" if AMPERE else "no bfloat16 in hardware: these runs will be slow")

In [ ]:
COMPILE = "--compile" if AMPERE else ""
SMALL = f"--web {COMPILE} --layers 6 --heads 6 --embd 384 --block 1024 --batch 16 --accum 4"
!torchrun --standalone --nproc_per_node 1 pretrain_ddp.py 10 {SMALL} --name colab-attention
!torchrun --standalone --nproc_per_node 1 pretrain_ddp.py 10 {SMALL} --hyena --name colab-hyena

## Following the numbers
The episode's half-hour models, from the Hub, on one text: attention's head 2 in block 3, and a
Hyena channel's gates, values and filter, with the last output worked out term by term.

In [ ]:
fetch("small-attention-1024", "small-hyena-1024")

In [ ]:
!python flow.py checkpoints/small-attention-1024/model.pt checkpoints/small-hyena-1024/model.pt "Marie Curie was born in Warsaw. Marie Curie was born in" --layer 3 --head 2

## The night's models
Episode 19's attention model, and the episode's Hyena and hybrid (attention in blocks 3 and 9),
110M parameters each, trained on 3.6 billion tokens of FineWeb-Edu: the loss by position on long
pages, random tokens twice, pass keys and writing speed, then HellaSwag and bits per character.

In [ ]:
fetch("pretrain-web", "web-hyena", "web-hybrid")

In [ ]:
!python hyena_eval.py checkpoints/pretrain-web/model.pt checkpoints/web-hyena/model.pt checkpoints/web-hybrid/model.pt

In [ ]:
!python hellaswag.py checkpoints/web-hyena/model.pt
!python hellaswag.py checkpoints/web-hybrid/model.pt

In [ ]:
!python bits.py checkpoints/pretrain-web/model.pt checkpoints/web-hyena/model.pt checkpoints/web-hybrid/model.pt --web

## Recall
Multi-query associative recall, from the Zoology paper: two blocks of each kind, 64 channels,
trained on 4 pairs and then up to 16, and scored on each. The episode ran 30,000 steps at three
widths and three learning rates; this is 6,000 steps at one.

In [ ]:
!python recall.py --widths 64 --pairs 4 8 16 --steps 6000